# GAN-RL Protein Function Prediction
**Goal:** Beat ProtHGT-ESM2 Biological Process Fmax (baseline: 0.7489)

**Workflow:** Cell 1 → Cell 11 in order. Checkpoints are saved to Drive after each phase.

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

In [ ]:
# ── Cell 2: Install Dependencies ───────────────────────────────────────────────
# Run this cell once per Colab session. Takes ~3-5 minutes.
import subprocess, sys

print('Installing PyTorch + CUDA 11.8 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'torch==2.1.0', 'torchvision', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu118', '-q'], check=True)

print('Installing PyG ...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'torch_geometric==2.4.0', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'torch_scatter', 'torch_sparse', 'torch_cluster',
    '-f', 'https://data.pyg.org/whl/torch-2.1.0+cu118.html', '-q'], check=True)

print('Installing other deps ...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'networkx==3.3', 'pyyaml==6.0.1', 'obonet==1.1.0',
    'scikit-learn==1.4.2', 'tqdm==4.66.4', 'pandas==2.2.2',
    'matplotlib==3.9.0', 'tensorboard==2.14.0', '-q'], check=True)

print('All dependencies installed.')

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/'
CHECKPOINT_DIR = '/content/drive/MyDrive/Poster code/checkpoints/'
LOG_CSV      = os.path.join(CHECKPOINT_DIR, 'training_log.csv')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Quick sanity check
esm2_dir = os.path.join(DRIVE_ROOT, 'alternative_protein_embeddings/esm2/')
expected = ['prothgt-train-graph.pt', 'prothgt-val-graph.pt', 'prothgt-test-graph.pt']
for f in expected:
    path = os.path.join(esm2_dir, f)
    exists = os.path.exists(path)
    print(f'  {f}: {"OK" if exists else "MISSING"}')
    if not exists:
        print(f'  ERROR: Could not find {path}')
        print(f'  Check that DRIVE_ROOT is correct: {DRIVE_ROOT}')

In [ ]:
# ── Cell 4: Clone Repository ───────────────────────────────────────────────────
import subprocess, sys, os

REPO_DIR = '/content/Prot_B_poster'
REPO_URL = 'https://github.com/Drjay806/Prot_B_poster.git'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Repo ready at {REPO_DIR}')
print('Contents:', os.listdir(REPO_DIR))

In [ ]:
# ── Cell 5: Load Config ────────────────────────────────────────────────────────
import yaml, os

config_path = os.path.join(REPO_DIR, 'configs/default.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Override paths with what we set above
cfg['data']['drive_root'] = DRIVE_ROOT
cfg['data']['checkpoint_dir'] = CHECKPOINT_DIR

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Target ontology:', cfg['data']['ontology'])
print('Target node type:', cfg['data']['target_type'])

In [ ]:
# ── Cell 6: Load Data + Build Ancestor Table ───────────────────────────────────
from src.data.loader import load_prothgt_splits
from src.data.go_hierarchy import build_ancestor_table

splits = load_prothgt_splits(
    drive_root=cfg['data']['drive_root'],
    ontology=cfg['data']['ontology'],
    device=DEVICE,
)
train_data, val_data, test_data = splits.train, splits.val, splits.test
target_type = splits.target_type

print(f'\nNode types: {train_data.node_types}')
print(f'Edge types: {train_data.edge_types}')
print(f'Proteins:   {train_data["Protein"].x.shape[0]:,}')
print(f'GO terms:   {train_data[target_type].x.shape[0]:,}')

print('\nBuilding GO hierarchy ancestor table ...')
ancestor_table = build_ancestor_table(train_data, target_type=target_type, cache=True)
print(f'Ancestor table: {len(ancestor_table):,} GO terms indexed')

In [ ]:
# ── Cell 7: Initialise Models ──────────────────────────────────────────────────
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.models.distmult import DistMult
from src.models.reward import RewardModule
from src.utils.logger import TrainingLogger

encoder      = CompGCN(train_data, cfg).to(DEVICE)
generator    = Generator(cfg).to(DEVICE)
discriminator = Discriminator(cfg).to(DEVICE)
distmult     = DistMult(hidden_dim=cfg['distmult']['hidden_dim']).to(DEVICE)
reward_module = RewardModule(cfg, distmult, discriminator).to(DEVICE)

total_params = sum(p.numel() for p in encoder.parameters()) + \
               sum(p.numel() for p in generator.parameters()) + \
               sum(p.numel() for p in discriminator.parameters())
print(f'Total trainable parameters: {total_params:,}')

logger = TrainingLogger(
    log_dir='/tmp/runs',
    csv_path=LOG_CSV,
)
print('Logger ready. Run `%load_ext tensorboard` then `%tensorboard --logdir /tmp/runs` to monitor.')

In [ ]:
# ── Cell 8: Phase 1 — Pre-training ────────────────────────────────────────────
# Trains CompGCN encoder with 4 losses (MSE, cosine, ranking, MMD).
# Generator/Discriminator are frozen during this phase.
# Expected: ~30-40 min on T4 for 50 epochs.

from src.training.pretrain import pretrain

encoder = pretrain(
    encoder=encoder,
    train_data=train_data,
    val_data=val_data,
    cfg=cfg,
    device=DEVICE,
    logger=logger,
)

# Save encoder checkpoint to Drive
import torch, os
ckpt_path = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained.pt')
torch.save(encoder.state_dict(), ckpt_path)
print(f'Saved pretrained encoder → {ckpt_path}')

In [ ]:
# ── Cell 8b: Plot Phase 1 Curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt, os

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
history = logger.history

def plot_metric(ax, key, label, color='steelblue'):
    if key in history:
        steps, vals = zip(*history[key])
        ax.plot(steps, vals, color=color, linewidth=1.5)
        ax.set_title(label); ax.set_xlabel('Step'); ax.grid(alpha=0.3)

for ax, (key, lbl, col) in zip(axes[:3], [
    ('loss/total', 'Total Pretrain Loss', 'steelblue'),
    ('val/cosine_similarity', 'Val Cosine Similarity', 'green'),
    ('embed/protein_norm_mean', 'Protein Emb Norm', 'orange'),
]):
    plot_metric(ax, key, lbl, col)

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'pretrain_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

In [ ]:
# ── Cell 9: Phase 2 — Adversarial Training ────────────────────────────────────
# GAN training: Discriminator updated 2x per Generator update.
# Expected: ~60-90 min on T4 for 100 epochs.

from src.training.adversarial import train_adversarial

encoder, generator, discriminator = train_adversarial(
    encoder=encoder,
    generator=generator,
    discriminator=discriminator,
    distmult=distmult,
    train_data=train_data,
    val_data=val_data,
    ancestor_table=ancestor_table,
    cfg=cfg,
    device=DEVICE,
    logger=logger,
)

ckpt_path = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')
torch.save({
    'encoder': encoder.state_dict(),
    'generator': generator.state_dict(),
    'discriminator': discriminator.state_dict(),
}, ckpt_path)
print(f'Saved adversarial checkpoint → {ckpt_path}')

In [ ]:
# ── Cell 9b: Plot Phase 2 Curves ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (key, lbl, col) in zip(axes, [
    ('loss/disc_total',    'Discriminator Loss',       'crimson'),
    ('loss/gen',           'Generator Loss',            'steelblue'),
    ('disc/real_acc',      'D Accuracy (Real)',         'green'),
    ('disc/fake_acc',      'D Accuracy (Fake)',         'darkorange'),
    ('reward/distmult_mean', 'DistMult Score (mean)',   'purple'),
    ('val/fmax_bp',        'Val Fmax (BP)',             'black'),
]):
    plot_metric(ax, key, lbl, col)

# Reference line for ProtHGT baseline
if 'val/fmax_bp' in history:
    steps, _ = zip(*history['val/fmax_bp'])
    axes[5].axhline(0.7489, color='red', linestyle='--', label='ProtHGT baseline')
    axes[5].legend()

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'adversarial_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

In [ ]:
# ── Cell 10: Phase 3 — RL Fine-tuning ────────────────────────────────────────
# REINFORCE with GO hierarchy penalty. Semantic reward grows via curriculum.
# Best val Fmax checkpoint is auto-saved to Drive.
# Expected: ~30-45 min on T4 for 50 epochs.

from src.training.rl_trainer import train_rl

encoder, generator = train_rl(
    encoder=encoder,
    generator=generator,
    distmult=distmult,
    reward_module=reward_module,
    train_data=train_data,
    val_data=val_data,
    ancestor_table=ancestor_table,
    cfg=cfg,
    device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR,
    logger=logger,
)

ckpt_path = os.path.join(CHECKPOINT_DIR, 'rl_final.pt')
torch.save({'encoder': encoder.state_dict(), 'generator': generator.state_dict()}, ckpt_path)
print(f'Saved final RL checkpoint → {ckpt_path}')

In [ ]:
# ── Cell 10b: Plot Phase 3 Curves ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (key, lbl, col) in zip(axes, [
    ('reward/total_mean',       'Mean Reward',              'steelblue'),
    ('reward/hierarchy_penalty','Hierarchy Penalty',         'crimson'),
    ('reward/semantic',         'Semantic Reward',           'green'),
    ('curriculum/w3',           'Semantic Weight w3(t)',     'orange'),
    ('grad/gen_norm',           'Generator Grad Norm',       'purple'),
    ('val/fmax_bp',             'Val Fmax (BP)',             'black'),
]):
    plot_metric(ax, key, lbl, col)

if 'val/fmax_bp' in history:
    axes[5].axhline(0.7489, color='red', linestyle='--', label='ProtHGT baseline')
    axes[5].legend()

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'rl_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

In [ ]:
# ── Cell 11: Final Evaluation on Test Split ───────────────────────────────────
# Uses exact ProtHGT test split with GO hierarchy propagation.
# All 5 metrics matching ProtHGT's evaluation protocol.

from src.evaluation.metrics import evaluate_all

print('\n=== FINAL TEST SET EVALUATION ===')
results = evaluate_all(
    encoder=encoder,
    generator=generator,
    distmult=distmult,
    data=test_data,
    ancestor_table=ancestor_table,
    target_type=target_type,
    cfg=cfg,
    device=DEVICE,
    baseline_fmax=0.7489,  # ProtHGT-ESM2 BP Fmax
)

# Save results to Drive
import json
results_path = os.path.join(CHECKPOINT_DIR, 'test_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved → {results_path}')

logger.close()

In [ ]:
# ── Optional: TensorBoard ──────────────────────────────────────────────────────
# Run this cell at any time to open TensorBoard and see live training curves.
%load_ext tensorboard
%tensorboard --logdir /tmp/runs/

In [ ]:
# ── Optional: Resume from Checkpoint ──────────────────────────────────────────
# If Colab disconnects mid-training, use this cell to reload from the last checkpoint.

RESUME_PHASE = 'adversarial'   # 'pretrain' | 'adversarial' | 'rl'
RESUME_PATH = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')  # adjust as needed

ckpt = torch.load(RESUME_PATH, map_location=DEVICE)

if RESUME_PHASE == 'pretrain':
    encoder.load_state_dict(ckpt)
    print('Loaded pretrained encoder.')
elif RESUME_PHASE in ('adversarial', 'rl'):
    encoder.load_state_dict(ckpt['encoder'])
    generator.load_state_dict(ckpt['generator'])
    if 'discriminator' in ckpt:
        discriminator.load_state_dict(ckpt['discriminator'])
    print(f'Loaded {RESUME_PHASE} checkpoint.')